# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/innouguru/flyrank-intenship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

# Retrieve the Hugging Face token stored in Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Check whether the token was successfully loaded
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [2]:
import duckdb        # Import DuckDB for working with data using SQL

con = duckdb.connect()      # Create an in-memory DuckDB connection

# Create a Hugging Face secret in DuckDB
con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)

rel = "hf://datasets/FlyRank/internship-warehouse"

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method:** Random Forest Classifier

**Why:**

- It combines predictions from multiple decision trees.
- It can learn relationships between multiple features.
- Week 4 showed that CTR should be interpreted relative to search position.
- Therefore, Random Forest is a reasonable method for learning these relationships and producing a ranking for refresh review.
- Its predicted probability can be used as the opportunity score.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
# Checking the time span of the warehouse data
date_range = con.sql(f"""
    SELECT
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date,
        COUNT(DISTINCT report_date) AS number_of_days
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
""").df()

date_range

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,first_date,last_date,number_of_days
0,2025-01-27,2026-06-30,520


In [4]:
# Checking the monthly coverage
monthly_counts = con.sql(f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,
        COUNT(*) AS rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_pages
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    GROUP BY 1
    ORDER BY 1
""").df()

monthly_counts

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,rows,clients,content_pages
0,2025-01-01,1297,2,476
1,2025-02-01,75985,3,5903
2,2025-03-01,167859,4,10374
3,2025-04-01,285114,4,13046
4,2025-05-01,349923,4,14887
5,2025-06-01,329201,9,16399
6,2025-07-01,469794,16,27945
7,2025-08-01,704962,15,37204
8,2025-09-01,845813,23,53127
9,2025-10-01,2165471,31,110339


In [5]:
# Count how many unique content pages appear in both March and April 2026.
# This tells us whether March pages can realistically be followed into the
# next month for an out-of-time evaluation.

march_april_overlap = con.sql(f"""
    WITH march_pages AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE report_date >= DATE '2026-03-01'
          AND report_date < DATE '2026-04-01'
    ),

    april_pages AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet'
        )
        WHERE report_date >= DATE '2026-04-01'
          AND report_date < DATE '2026-05-01'
    )

    SELECT
        COUNT(*) AS march_pages,
        COUNT(april.content_hash_id) AS pages_also_in_april,
        ROUND(
            100.0 * COUNT(april.content_hash_id) / COUNT(*),
            2
        ) AS pct_march_pages_also_in_april
    FROM march_pages march
    LEFT JOIN april_pages april
        ON march.client_hash_id = april.client_hash_id
       AND march.content_hash_id = april.content_hash_id
""").df()

march_april_overlap

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_pages,pages_also_in_april,pct_march_pages_also_in_april
0,331437,331436,100.0


In [6]:
# Check how many monthly page-client observations have both
# Google Search Console (GSC) and Google Analytics 4 (GA4) data available.
#
# We are doing this before filtering so we can see how much data
# will remain for the modeling experiment.

availability_check = con.sql(f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,

        -- Count all monthly page-client observations.
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id) AS pages,

        -- Count observations where both data sources are available.
        COUNT(DISTINCT CASE
            WHEN gsc_data_available = TRUE
             AND ga4_data_available = TRUE
            THEN client_hash_id || '|' || content_hash_id
        END) AS pages_with_both

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )

    GROUP BY 1
    ORDER BY 1
""").df()

# Calculate the percentage of page-client observations
# that have both GSC and GA4 data available.
availability_check["pct_with_both"] = (
    100 * availability_check["pages_with_both"]
    / availability_check["pages"]
)

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month,pages,pages_with_both,pct_with_both
0,2025-01-01,476,0,0.000000
1,2025-02-01,5903,0,0.000000
2,2025-03-01,10374,0,0.000000
3,2025-04-01,13046,0,0.000000
4,2025-05-01,14887,0,0.000000
5,2025-06-01,16399,0,0.000000
6,2025-07-01,27945,0,0.000000
7,2025-08-01,37204,0,0.000000
8,2025-09-01,53127,0,0.000000
9,2025-10-01,110339,3828,3.469308


In [7]:
# Check which March content pages are still present in April.
# We need this because a March recommendation can only be evaluated
# in April if we can identify the same page in the next month.

march_april_continuity = con.sql(
    f"""
    WITH march_pages AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )
        WHERE month = '2026-03'
    ),

    april_pages AS (
        SELECT DISTINCT
            client_hash_id,
            content_hash_id
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )
        WHERE month = '2026-04'
    )

    SELECT
        COUNT(*) AS march_pages,

        -- Count March pages that can also be found in April.
        COUNT(*) FILTER (
            WHERE a.content_hash_id IS NOT NULL
        ) AS pages_in_april,

        -- Calculate the percentage of March pages
        -- that continue into the following month.
        ROUND(
            100.0 * COUNT(*) FILTER (
                WHERE a.content_hash_id IS NOT NULL
            ) / COUNT(*),
            2
        ) AS continuity_pct

    FROM march_pages m

    LEFT JOIN april_pages a
        ON m.client_hash_id = a.client_hash_id
        AND m.content_hash_id = a.content_hash_id
    """
).df()

march_april_continuity

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_pages,pages_in_april,continuity_pct
0,331437,331436,100.0


In [8]:
# Create the April outcome label.
# A page is considered a positive outcome if its April CTR
# is at least 30% below the median CTR of pages in the
# same April position peer group.

april_outcome = con.sql(
    f"""
    WITH april_base AS (
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,

            -- Calculate April CTR as a percentage.
            CASE
                WHEN gsc_impressions > 0
                THEN (gsc_clicks * 100.0) / gsc_impressions
                ELSE NULL
            END AS ctr,

            -- Create the same 10-position peer groups
            -- used in the Week 4 baseline.
            FLOOR((gsc_avg_position - 1) / 10) * 10 + 1
                AS position_start

        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        -- April is the future evaluation window.
        WHERE month = '2026-04'

          -- Use the same data-availability requirement
          -- used by the Week 4 baseline.
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE

          -- Exclude observations without a valid position.
          AND gsc_avg_position >= 1

          -- CTR must be calculable.
          AND gsc_impressions > 0
    ),

    april_peers AS (
        SELECT
            *,

            -- Calculate the median CTR for each April
            -- position-peer group.
            MEDIAN(ctr) OVER (
                PARTITION BY position_start
            ) AS peer_median_ctr,

            -- Count observations in each peer group.
            COUNT(*) OVER (
                PARTITION BY position_start
            ) AS peer_count

        FROM april_base
    )

    SELECT
        client_hash_id,
        content_hash_id,
        ctr AS april_ctr,
        gsc_avg_position AS april_position,
        peer_median_ctr,
        peer_count,

        -- Positive outcome:
        -- April CTR is at least 30% below the April
        -- position-peer median.
        CASE
            WHEN peer_count >= 15
             AND ctr <= peer_median_ctr * 0.70
            THEN 1
            ELSE 0
        END AS april_ctr_problem

    FROM april_peers
    """
).df()

# Display the first few rows to inspect the result.
april_outcome.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,april_ctr,april_position,peer_median_ctr,peer_count,april_ctr_problem
0,client_9958f0a7ae1df715,content_01abeb8b40591eec,0.0,15.133333,0.0,76285,1
1,client_9958f0a7ae1df715,content_3c3b575d53a71932,0.0,19.464286,0.0,76285,1
2,client_9958f0a7ae1df715,content_0f29eb1eece11b2f,0.0,13.892857,0.0,76285,1
3,client_9958f0a7ae1df715,content_94798aa612b0e422,0.0,11.428571,0.0,76285,1
4,client_9958f0a7ae1df715,content_5147a94966f9a69b,0.0,17.846154,0.0,76285,1


In [9]:
# Check the number of April pages and the proportion
# that meet our future CTR-problem definition.

print("April pages evaluated:", len(april_outcome))

print(
    "April CTR-problem rate:",
    round(april_outcome["april_ctr_problem"].mean(), 3)
)

# Check how many pages have enough position peers
# to receive a valid peer benchmark.
print(
    "Pages with >=15 peer observations:",
    (april_outcome["peer_count"] >= 15).sum()
)

April pages evaluated: 455813
April CTR-problem rate: 0.496
Pages with >=15 peer observations: 455754


In [10]:
# Check how the April CTR-problem rate varies across
# the 10-position peer groups.

april_outcome["position_bucket"] = (
    ((april_outcome["april_position"] - 1) // 10) * 10 + 1
)

position_outcome_check = (
    april_outcome
    .groupby("position_bucket")
    .agg(
        pages=("content_hash_id", "count"),
        ctr_problem_rate=("april_ctr_problem", "mean")
    )
    .reset_index()
)

# Convert the outcome rate to a percentage for readability.
position_outcome_check["ctr_problem_rate"] = (
    position_outcome_check["ctr_problem_rate"] * 100
).round(2)

position_outcome_check.head(15)

,position_bucket,pages,ctr_problem_rate
0,1.0,286903,43.67
1,11.0,76285,51.75
2,21.0,46871,58.15
3,31.0,26510,66.66
4,41.0,10847,78.48
5,51.0,3917,88.77
6,61.0,1903,93.06
7,71.0,1223,96.48
8,81.0,809,99.13
9,91.0,469,99.15


In [11]:
# Calculate how each page's April CTR compares with
# the median CTR of its April position peers.
#
# A value of:
#   1.00 = equal to the peer median
#   0.70 = 30% below the peer median
#   0.50 = 50% below the peer median
#
# This is the same relative-CTR idea used by the Week 4 rule.

april_outcome["relative_ctr"] = (
    april_outcome["april_ctr"]
    / april_outcome["peer_median_ctr"]
)

# Create the April evaluation outcome using the same
# 30% below-peer threshold from Week 4.
april_outcome["april_ctr_below_peer"] = (
    (april_outcome["peer_count"] >= 15)
    & (april_outcome["relative_ctr"] <= 0.70)
).astype(int)

# Inspect the distribution of the relative CTR.
april_outcome["relative_ctr"].describe()

,relative_ctr
count,3.551260e+05
mean,inf
std,NaN
min,0.000000e+00
25%,0.000000e+00
50%,1.667845e+00
75%,8.232558e+00
max,inf


In [12]:
# Check how many April pages have a position-peer median CTR
# equal to zero.
#
# A relative CTR cannot be calculated for these pages because
# the peer benchmark would be zero.

zero_peer_median = (
    april_outcome["peer_median_ctr"] == 0
)

print(
    "Pages with zero peer median CTR:",
    zero_peer_median.sum()
)

print(
    "Percentage of April pages:",
    round(zero_peer_median.mean() * 100, 2),
)


Pages with zero peer median CTR: 168910
Percentage of April pages: 37.06


In [13]:
# Identify the April position groups where the median CTR is zero.
# These groups cannot provide a meaningful relative-CTR benchmark.

zero_median_groups = (
    april_outcome[
        april_outcome["peer_median_ctr"] == 0
    ]
    .groupby("april_position")
    .agg(
        pages=("content_hash_id", "count"),
        zero_ctr_pages=("april_ctr", lambda x: (x == 0).sum())
    )
    .reset_index()
)

# Calculate the percentage of pages with zero CTR
# inside each zero-median position group.
zero_median_groups["zero_ctr_pct"] = (
    100
    * zero_median_groups["zero_ctr_pages"]
    / zero_median_groups["pages"]
)

zero_median_groups.head(20)

,april_position,pages,zero_ctr_pages,zero_ctr_pct
0,11.000000,828,714,86.231884
1,11.001337,1,0,0.000000
2,11.002865,1,1,100.000000
3,11.003322,1,0,0.000000
4,11.003356,1,0,0.000000
5,11.004255,1,1,100.000000
6,11.004310,1,0,0.000000
7,11.004386,1,1,100.000000
8,11.004762,1,1,100.000000
9,11.004902,1,0,0.000000


In [14]:
# Recreate the 10-position bucket used by the Week 4 baseline.
# This groups positions as:
# 1-10, 11-20, 21-30, etc.

april_outcome["position_start"] = (
    ((april_outcome["april_position"] - 1) // 10) * 10 + 1
)

# Identify the position buckets where the peer median CTR is zero.
zero_median_groups = (
    april_outcome[
        april_outcome["peer_median_ctr"] == 0
    ]
    .groupby("position_start")
    .agg(
        pages=("content_hash_id", "count"),
        zero_ctr_pages=(
            "april_ctr",
            lambda x: (x == 0).sum()
        )
    )
    .reset_index()
)

# Calculate the percentage of pages with zero CTR
# within each zero-median position bucket.
zero_median_groups["zero_ctr_pct"] = (
    100
    * zero_median_groups["zero_ctr_pages"]
    / zero_median_groups["pages"]
)

zero_median_groups


,position_start,pages,zero_ctr_pages,zero_ctr_pct
0,11.0,76285,39478,51.750672
1,21.0,46871,27257,58.153229
2,31.0,26510,17672,66.661637
3,41.0,10847,8513,78.482530
4,51.0,3917,3477,88.766913
5,61.0,1903,1771,93.063584
6,71.0,1223,1180,96.484056
7,81.0,809,802,99.134734
8,91.0,469,465,99.147122
9,101.0,17,15,88.235294


In [15]:
# List the months available in the warehouse.
# We need consecutive months because the model will use month T
# to predict whether the page improves in month T+1.

available_months = con.sql(
    f"""
    SELECT DISTINCT
        DATE_TRUNC('month', report_date) AS month
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    ORDER BY month
    """
).df()

# Display the available months.
available_months

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,month
0,2025-01-01
1,2025-02-01
2,2025-03-01
3,2025-04-01
4,2025-05-01
5,2025-06-01
6,2025-07-01
7,2025-08-01
8,2025-09-01
9,2025-10-01


In [16]:
# Define the historical training window.
# March and April 2026 are deliberately excluded because
# they will be used for the final evaluation.

training_data = con.sql(
    f"""
    SELECT
        DATE_TRUNC('month', report_date) AS month,
        client_hash_id,
        content_hash_id,

        -- Total monthly search impressions.
        SUM(gsc_impressions) AS gsc_impressions,

        -- Total monthly search clicks.
        SUM(gsc_clicks) AS gsc_clicks,

        -- Impression-weighted average search position.
        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN SUM(gsc_avg_position * gsc_impressions)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position,

        -- Total monthly pageviews.
        SUM(ga4_pageviews) AS ga4_pageviews,

        -- Total monthly engaged sessions.
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    -- Use only months that will be used to construct
    -- the Random Forest training examples.
    WHERE report_date >= '2025-10-01'
      AND report_date < '2026-03-01'

      -- Match the Week 4 data availability requirement.
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE

    GROUP BY
        DATE_TRUNC('month', report_date),
        client_hash_id,
        content_hash_id
    """
).df()

# Check how much data was extracted.
print("Training rows:", len(training_data))

# Check the months included in the training data.
print("\nMonths:")
print(training_data["month"].sort_values().unique())

# Check the available columns.
print("\nColumns:")
print(training_data.columns.tolist())

# Display a few records.
training_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training rows: 74213

Months:
<DatetimeArray>
['2025-10-01 00:00:00', '2025-11-01 00:00:00', '2025-12-01 00:00:00',
 '2026-01-01 00:00:00', '2026-02-01 00:00:00']
Length: 5, dtype: datetime64[us]

Columns:
['month', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


,month,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,2025-10-01,client_ff644d8251367cbb,content_ad81eee3c0435e0b,1183.0,8.0,4.128487,9.0,0.0
1,2025-10-01,client_ff644d8251367cbb,content_2a13e3ba9d7098a3,16.0,1.0,4.312500,1.0,0.0
2,2025-10-01,client_ff644d8251367cbb,content_b7d9d2e9fbb172ed,1012.0,3.0,4.403162,3.0,0.0
3,2025-10-01,client_ff644d8251367cbb,content_258985ae32b53385,3.0,1.0,10.333333,1.0,0.0
4,2025-10-01,client_ff644d8251367cbb,content_169171bfe15df2a1,1.0,1.0,10.000000,1.0,0.0


In [17]:
# Count records with an invalid search position.
# Week 4 excluded observations where the average position was 0.

invalid_position = (
    training_data["gsc_avg_position"] <= 0
).sum()

print("Rows with invalid position:", invalid_position)

# Remove records without a valid search position.
# This keeps the training data consistent with the Week 4 baseline.

training_data = training_data[
    training_data["gsc_avg_position"] >= 1
].copy()

# Check the remaining number of records.
print("Training rows after position filter:", len(training_data))

# Confirm that no invalid positions remain.
print(
    "Invalid positions remaining:",
    (training_data["gsc_avg_position"] < 1).sum()
)

Rows with invalid position: 1086
Training rows after position filter: 71526
Invalid positions remaining: 0


In [18]:
# Calculate CTR for each page-month observation.
#
# CTR is calculated from monthly clicks and impressions,
# consistent with the Week 4 baseline.

training_data["ctr"] = (
    training_data["gsc_clicks"] * 100.0
    / training_data["gsc_impressions"]
)

# Create the same 10-position peer groups used in Week 4.
#
# Examples:
#   1.0–10.999  -> group starting at 1
#   11.0–20.999 -> group starting at 11
#   21.0–30.999 -> group starting at 21
#
# We keep the group based on the page's position at the
# time the recommendation would have been made.

training_data["position_start"] = (
    ((training_data["gsc_avg_position"] - 1) // 10) * 10 + 1
)

# Calculate the median CTR of each position peer group.
#
# Peers are restricted to the same:
#   - month
#   - client
#   - position group
#
# This prevents pages from different clients or months
# from being compared with each other.

peer_median = (
    training_data
    .groupby(
        ["month", "client_hash_id", "position_start"]
    )["ctr"]
    .transform("median")
)

# Add the peer median to the training data.
training_data["peer_median_ctr"] = peer_median

# Measure how many observations have a valid peer median.
valid_peer_median = training_data["peer_median_ctr"].notna().sum()

print("Training observations:", len(training_data))
print("Observations with peer median:", valid_peer_median)

# Display a few examples so we can inspect the calculation.
training_data[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "gsc_avg_position",
        "ctr",
        "position_start",
        "peer_median_ctr"
    ]
].head(10)

Training observations: 71526
Observations with peer median: 71526


,month,client_hash_id,content_hash_id,gsc_avg_position,ctr,position_start,peer_median_ctr
0,2025-10-01,client_ff644d8251367cbb,content_ad81eee3c0435e0b,4.128487,0.676247,1.0,1.764706
1,2025-10-01,client_ff644d8251367cbb,content_2a13e3ba9d7098a3,4.312500,6.250000,1.0,1.764706
2,2025-10-01,client_ff644d8251367cbb,content_b7d9d2e9fbb172ed,4.403162,0.296443,1.0,1.764706
3,2025-10-01,client_ff644d8251367cbb,content_258985ae32b53385,10.333333,33.333333,1.0,1.764706
4,2025-10-01,client_ff644d8251367cbb,content_169171bfe15df2a1,10.000000,100.000000,1.0,1.764706
5,2025-10-01,client_ff644d8251367cbb,content_5390a6ce2b6498a8,7.211712,0.450450,1.0,1.764706
6,2025-10-01,client_9958f0a7ae1df715,content_4cec18f637b4c858,5.794872,0.000000,1.0,0.947867
7,2025-10-01,client_9958f0a7ae1df715,content_6edb421d967970be,4.674544,0.322234,1.0,0.947867
8,2025-10-01,client_9958f0a7ae1df715,content_9b3decff0d7e6690,7.185185,1.851852,1.0,0.947867
9,2025-10-01,client_9958f0a7ae1df715,content_dc84473ea94d2c61,8.835902,0.725295,1.0,0.947867


In [19]:
# Count the number of pages in each position-peer group.
#
# Peers are defined within the same month, client, and
# 10-position group.

peer_group_columns = [
    "month",
    "client_hash_id",
    "position_start"
]

training_data["peer_count"] = (
    training_data
    .groupby(peer_group_columns)["content_hash_id"]
    .transform("count")
)

# Display the distribution of peer-group sizes.
print(training_data["peer_count"].describe())

# Count observations that have at least 15 peers.
pages_with_15_peers = (
    training_data["peer_count"] >= 15
).sum()

print(
    "\nPages with >=15 peer observations:",
    pages_with_15_peers
)

# Calculate the percentage of observations with
# at least 15 peer observations.
pct_with_15_peers = (
    pages_with_15_peers / len(training_data) * 100
)

print(
    "Percentage with >=15 peers:",
    round(pct_with_15_peers, 2)
)

count    71526.000000
mean      2307.494701
std       1570.653111
min          1.000000
25%        718.000000
50%       2075.000000
75%       3751.000000
max       4645.000000
Name: peer_count, dtype: float64

Pages with >=15 peer observations: 70671
Percentage with >=15 peers: 98.8


In [20]:
# Keep only observations with enough position-peer observations
# to make the peer comparison reasonably reliable.
training_data = training_data[
    training_data["peer_count"] >= 15
].copy()

# Reproduce the Week 4 rule:
# a page is a candidate when its CTR is below the median CTR
# of pages in the same position peer group.
training_data["ctr_below_position_peers"] = (
    training_data["ctr"] < training_data["peer_median_ctr"]
)

# Check how many historical observations would have been
# selected by the Week 4 rule.
print(
    "CTR candidates:",
    training_data["ctr_below_position_peers"].sum()
)

print(
    "Candidate rate:",
    round(
        training_data["ctr_below_position_peers"].mean() * 100,
        2
    ),
    "%"
)

CTR candidates: 28486
Candidate rate: 40.31 %


In [21]:
# Build the future-month outcome data.
#
# For each page in the training period, we will later compare
# its current-month status with its next-month status.

future_data = con.sql(
    f"""
    WITH base AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,

            -- Calculate CTR as a percentage.
            CASE
                WHEN gsc_impressions > 0
                THEN (gsc_clicks * 100.0) / gsc_impressions
                ELSE NULL
            END AS ctr,

            -- Create the same 10-position peer groups used
            -- in the Week 4 rule.
            FLOOR((gsc_avg_position - 1) / 10) * 10 + 1
                AS position_start

        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        -- Use only the months immediately following
        -- our five training months.
        WHERE month >= '2025-11'
          AND month <= '2026-03'

          -- Match the data-availability requirement from Week 4.
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE

          -- Require a valid search position.
          AND gsc_avg_position >= 1
    ),

    peer_stats AS (
        SELECT
            month,
            position_start,

            -- Calculate the median CTR for pages in the
            -- same month and position peer group.
            MEDIAN(ctr) AS peer_median_ctr,

            -- Count the number of observations in the peer group.
            COUNT(*) AS peer_count

        FROM base

        -- CTR must exist before calculating peer statistics.
        WHERE ctr IS NOT NULL

        GROUP BY
            month,
            position_start
    )

    SELECT
        b.month,
        b.client_hash_id,
        b.content_hash_id,
        b.gsc_avg_position,
        b.ctr,
        b.position_start,
        p.peer_median_ctr,
        p.peer_count

    FROM base b

    -- Attach the peer statistics for the same future month
    -- and the same position group.
    INNER JOIN peer_stats p
        ON b.month = p.month
        AND b.position_start = p.position_start

    -- Keep only observations with a sufficiently large
    -- peer group, matching our >=15 rule.
    WHERE p.peer_count >= 15
      AND b.ctr IS NOT NULL

    ORDER BY
        b.month,
        b.client_hash_id,
        b.content_hash_id
    """
).df()

# Check the resulting future-month dataset.
print("Future observations:", len(future_data))
print("Months:", future_data["month"].unique())

# Display a few observations to verify the structure.
future_data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future observations: 745268
Months: ['2025-11' '2025-12' '2026-01' '2026-02' '2026-03']


,month,client_hash_id,content_hash_id,gsc_avg_position,ctr,position_start,peer_median_ctr,peer_count
0,2025-11,client_23a62021009f63c4,content_0013b0a02c45a7c8,28.880952,0.000000,21.0,0.000000,2862
1,2025-11,client_23a62021009f63c4,content_0013b0a02c45a7c8,21.809524,0.000000,21.0,0.000000,2862
2,2025-11,client_23a62021009f63c4,content_0017e6d53661a061,10.045455,0.000000,1.0,0.641026,65750
3,2025-11,client_23a62021009f63c4,content_0017e6d53661a061,7.828571,0.000000,1.0,0.641026,65750
4,2025-11,client_23a62021009f63c4,content_0017e6d53661a061,6.526882,1.075269,1.0,0.641026,65750


In [22]:
# Build one monthly observation for each content page.
#
# The warehouse contains daily observations, but our model
# operates at the page-month level.

future_monthly = con.sql(
    f"""
    WITH daily AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position

        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )

        -- Use the months immediately after our five
        -- training months.
        WHERE month >= '2025-11'
          AND month <= '2026-03'

          -- Use the same data-availability requirement as Week 4.
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE

          -- Exclude observations without a valid search position.
          AND gsc_avg_position >= 1
    ),

    monthly_pages AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,

            -- Aggregate clicks and impressions across the month.
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,

            -- Calculate the average search position for the month.
            AVG(gsc_avg_position) AS avg_position

        FROM daily

        GROUP BY
            month,
            client_hash_id,
            content_hash_id
    ),

    page_metrics AS (
        SELECT
            month,
            client_hash_id,
            content_hash_id,
            avg_position,

            -- Calculate monthly CTR from total clicks
            -- divided by total impressions.
            CASE
                WHEN impressions > 0
                THEN (clicks * 100.0) / impressions
                ELSE NULL
            END AS ctr,

            -- Create the same 10-position peer groups
            -- used in the Week 4 rule.
            FLOOR((avg_position - 1) / 10) * 10 + 1
                AS position_start

        FROM monthly_pages
    ),

    peer_stats AS (
        SELECT
            month,
            position_start,

            -- Calculate the median CTR for each
            -- month and position peer group.
            MEDIAN(ctr) AS peer_median_ctr,

            -- Count the number of pages in each peer group.
            COUNT(*) AS peer_count

        FROM page_metrics

        WHERE ctr IS NOT NULL

        GROUP BY
            month,
            position_start
    )

    SELECT
        p.month,
        p.client_hash_id,
        p.content_hash_id,
        p.avg_position AS gsc_avg_position,
        p.ctr,
        p.position_start,
        s.peer_median_ctr,
        s.peer_count

    FROM page_metrics p

    -- Attach the peer statistics for the same month
    -- and the same position group.
    INNER JOIN peer_stats s
        ON p.month = s.month
        AND p.position_start = s.position_start

    -- Keep only peer groups with at least 15 observations.
    WHERE s.peer_count >= 15
      AND p.ctr IS NOT NULL

    ORDER BY
        p.month,
        p.client_hash_id,
        p.content_hash_id
    """
).df()


# Verify that each page appears only once per month.
duplicate_check = (
    future_monthly
    .groupby(
        ["month", "client_hash_id", "content_hash_id"]
    )
    .size()
)

print(
    "Maximum observations per page-month:",
    duplicate_check.max()
)

print(
    "Future monthly observations:",
    len(future_monthly)
)

print(
    "Months:",
    future_monthly["month"].unique()
)

future_monthly.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Maximum observations per page-month: 1
Future monthly observations: 130091
Months: ['2025-11' '2025-12' '2026-01' '2026-02' '2026-03']


,month,client_hash_id,content_hash_id,gsc_avg_position,ctr,position_start,peer_median_ctr,peer_count
0,2025-11,client_23a62021009f63c4,content_0013b0a02c45a7c8,25.345238,0.000000,21.0,0.000000,662
1,2025-11,client_23a62021009f63c4,content_0017e6d53661a061,12.781149,0.130208,11.0,0.501463,2322
2,2025-11,client_23a62021009f63c4,content_0018b50e392faeb8,7.925000,0.000000,1.0,0.717489,10573
3,2025-11,client_23a62021009f63c4,content_001ccd954baa493f,20.625000,0.000000,11.0,0.501463,2322
4,2025-11,client_23a62021009f63c4,content_002ab5d74a38ddd7,7.791667,0.000000,1.0,0.717489,10573


In [23]:
import pandas as pd
# Make copies so we don't accidentally modify the original
# training and future datasets.

current_data = training_data.copy()
next_data = future_monthly.copy()

# Convert the month columns to datetime so we can calculate
# the following month reliably.

current_data["month"] = pd.to_datetime(current_data["month"])
next_data["month"] = pd.to_datetime(next_data["month"])

# Create the month in which we expect to observe the outcome.
#
# Example:
# October 2025 -> November 2025
# November 2025 -> December 2025

current_data["next_month"] = (
    current_data["month"] + pd.DateOffset(months=1)
)

# Keep only pages that were identified as CTR problems
# by the Week 4 rule.
#
# This means the Random Forest will learn which candidates
# are more likely to recover, rather than simply learning
# how to identify low-CTR pages.

current_candidates = current_data[
    current_data["ctr_below_position_peers"]
].copy()

print(
    "Current-month CTR candidates:",
    len(current_candidates)
)

# Select only the columns needed from the next-month data.
next_outcomes = next_data[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "ctr",
        "peer_median_ctr",
        "peer_count"
    ]
].copy()

# Rename the outcome columns so it is clear that they belong
# to the following month.

next_outcomes = next_outcomes.rename(
    columns={
        "month": "next_month",
        "ctr": "next_ctr",
        "peer_median_ctr": "next_peer_median_ctr",
        "peer_count": "next_peer_count"
    }
)

# Match each current-month candidate to the same page in
# the following month.

target_data = current_candidates.merge(
    next_outcomes,
    on=[
        "next_month",
        "client_hash_id",
        "content_hash_id"
    ],
    how="inner"
)

# Define recovery:
#
# The page was below its position peers in the current month,
# and in the next month its CTR reached or exceeded the
# next month's position-peer median.

target_data["future_ctr_improved"] = (
    target_data["next_ctr"]
    >= target_data["next_peer_median_ctr"]
)

# Convert the Boolean target to 0/1 for model training.

target_data["target"] = (
    target_data["future_ctr_improved"]
    .astype(int)
)

# Inspect the target distribution.

print(
    "Candidate observations with a future outcome:",
    len(target_data)
)

print(
    "Recovered pages:",
    target_data["target"].sum()
)

print(
    "Recovery rate:",
    round(target_data["target"].mean() * 100, 2),
    "%"
)

# Show the first few examples so we can manually verify
# that the target makes sense.

target_data[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "ctr",
        "peer_median_ctr",
        "next_month",
        "next_ctr",
        "next_peer_median_ctr",
        "target"
    ]
].head(10)

Current-month CTR candidates: 28486
Candidate observations with a future outcome: 23388
Recovered pages: 10420
Recovery rate: 44.55 %


,month,client_hash_id,content_hash_id,ctr,peer_median_ctr,next_month,next_ctr,next_peer_median_ctr,target
0,2025-10-01,client_ff644d8251367cbb,content_ad81eee3c0435e0b,0.676247,1.764706,2025-11-01,1.038537,0.717489,1
1,2025-10-01,client_ff644d8251367cbb,content_b7d9d2e9fbb172ed,0.296443,1.764706,2025-11-01,0.413793,0.717489,0
2,2025-10-01,client_ff644d8251367cbb,content_5390a6ce2b6498a8,0.450450,1.764706,2025-11-01,0.530504,0.717489,0
3,2025-10-01,client_9958f0a7ae1df715,content_4cec18f637b4c858,0.000000,0.947867,2025-11-01,0.954085,0.717489,1
4,2025-10-01,client_9958f0a7ae1df715,content_6edb421d967970be,0.322234,0.947867,2025-11-01,0.536439,0.717489,0
5,2025-10-01,client_9958f0a7ae1df715,content_dc84473ea94d2c61,0.725295,0.947867,2025-11-01,0.744681,0.717489,1
6,2025-10-01,client_9958f0a7ae1df715,content_79d8051bbc47e80b,0.642398,0.947867,2025-11-01,0.362319,0.717489,0
7,2025-10-01,client_9958f0a7ae1df715,content_29405703cd2ce3ea,0.437637,0.947867,2025-11-01,0.460829,0.717489,0
8,2025-10-01,client_9958f0a7ae1df715,content_b516da7950088e94,0.186654,0.947867,2025-11-01,0.227401,0.717489,0
9,2025-10-01,client_9958f0a7ae1df715,content_e9bef9725fd22c03,0.389105,0.947867,2025-11-01,0.189138,0.717489,0


In [24]:
print("Target distribution:")
print(target_data["target"].value_counts())

print("\nTarget proportion:")
print(target_data["target"].value_counts(normalize=True))

Target distribution:
target
0    12968
1    10420
Name: count, dtype: int64

Target proportion:
target
0    0.554472
1    0.445528
Name: proportion, dtype: float64


In [25]:
target_data.groupby("month").size()

,0
month,
2025-10-01,1508
2025-11-01,5184
2025-12-01,5166
2026-01-01,5795
2026-02-01,5735


In [26]:
target_data.groupby("month")["target"].agg(
    ["count", "sum", "mean"]
)

,count,sum,mean
month,,,
2025-10-01,1508,607,0.402520
2025-11-01,5184,2098,0.404707
2025-12-01,5166,2494,0.482772
2026-01-01,5795,2560,0.441760
2026-02-01,5735,2661,0.463993


In [27]:
# Split the labeled dataset chronologically.
# We use October 2025 through January 2026 for model training
# and hold February 2026 out as a later validation period.
# This prevents information from later months from entering training.

train_df = target_data[
    target_data["month"] < "2026-02-01"
].copy()

# Keep February 2026 completely separate for validation.
# The model will not see these observations during training.

val_df = target_data[
    target_data["month"] == "2026-02-01"
].copy()

# Check the number of observations assigned to each split.

print("Training rows:", len(train_df))
print("Validation rows:", len(val_df))

# Verify which months are actually included in the training set.
# This confirms that the split is chronological as intended.

print("\nTraining months:")
print(train_df["month"].unique())

# Verify the validation period separately.

print("\nValidation month:")
print(val_df["month"].unique())

Training rows: 17653
Validation rows: 5735

Training months:
<DatetimeArray>
['2025-10-01 00:00:00', '2025-11-01 00:00:00', '2025-12-01 00:00:00',
 '2026-01-01 00:00:00']
Length: 4, dtype: datetime64[us]

Validation month:
<DatetimeArray>
['2026-02-01 00:00:00']
Length: 1, dtype: datetime64[us]


In [28]:
# Define the five input features used by the Random Forest.
# These are the same raw performance signals used in the Week 4 analysis
# and are available at the decision moment.

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

# Separate the input features from the target for the training period.
# The target represents the observed future outcome and must not be included
# among the model's input features.

X_train = train_df[feature_cols].copy()
y_train = train_df["target"].copy()

# Create the corresponding feature matrix and target for the validation period.
# February observations remain completely unseen during model fitting.

X_val = val_df[feature_cols].copy()
y_val = val_df["target"].copy()

# Confirm the dimensions of the resulting datasets.

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

# Display the feature names so we can verify that no future-derived
# or rule-derived columns were accidentally included.

print("\nFeatures used by the model:")
print(feature_cols)

X_train shape: (17653, 5)
y_train shape: (17653,)
X_val shape: (5735, 5)
y_val shape: (5735,)

Features used by the model:
['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_engaged_sessions']


In [29]:
# Check whether any of the five model features contain missing values.
# This is important because the standard Random Forest implementation
# cannot train directly on NaN values.

print("Missing values in training features:")
print(X_train.isna().sum())

print("\nMissing values in validation features:")
print(X_val.isna().sum())

# Check whether the target contains any missing values.
# Every training and validation observation should have a known target.

print("\nMissing training targets:", y_train.isna().sum())
print("Missing validation targets:", y_val.isna().sum())

Missing values in training features:
gsc_impressions         0
gsc_clicks              0
gsc_avg_position        0
ga4_pageviews           0
ga4_engaged_sessions    0
dtype: int64

Missing values in validation features:
gsc_impressions         0
gsc_clicks              0
gsc_avg_position        0
ga4_pageviews           0
ga4_engaged_sessions    0
dtype: int64

Missing training targets: 0
Missing validation targets: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [30]:
from sklearn.ensemble import RandomForestClassifier

# Create the Random Forest classifier using the five raw performance signals.
# No preprocessing is required because the features are numeric, complete,
# and Random Forest does not require feature scaling.

rf_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

# Train the model only on the historical training observations.
# February 2026 remains completely unseen during training.

rf_model.fit(X_train, y_train)

print("Random Forest training complete.")

Random Forest training complete.


In [31]:
# Generate the model's probability that each February observation belongs
# to the positive class (target = 1).
# This probability will serve as the model's ranking score.

val_scores = rf_model.predict_proba(X_val)[:, 1]

# Add the model score to a copy of the validation data so we can inspect
# how the model ranks individual observations.

val_results = val_df.copy()
val_results["model_score"] = val_scores

# Sort observations from highest to lowest predicted probability.
# Pages at the top are the pages the model considers most likely
# to experience the positive future outcome.

val_results = val_results.sort_values(
    "model_score",
    ascending=False
)

# Display the highest-ranked observations.

val_results[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "model_score",
        "target"
    ]
].head(10)

,month,client_hash_id,content_hash_id,model_score,target
8933,2026-02-01,client_9958f0a7ae1df715,content_4dcfe9eefcadae59,0.985556,1
8916,2026-02-01,client_65de48885f4ef01b,content_083ddc844ce17ae9,0.983759,0
11131,2026-02-01,client_e547b89c05043229,content_8d495cf34f7eb996,0.983333,1
20461,2026-02-01,client_23a62021009f63c4,content_4505547f1b440f71,0.980000,0
20165,2026-02-01,client_e547b89c05043229,content_c4ff8ca15564f0b2,0.976667,0
11037,2026-02-01,client_ff644d8251367cbb,content_66838395d3e3026b,0.975000,0
9071,2026-02-01,client_ff644d8251367cbb,content_424bb4266b442cf7,0.974667,0
23239,2026-02-01,client_23a62021009f63c4,content_d9302b7787c172ab,0.973333,1
19830,2026-02-01,client_9958f0a7ae1df715,content_a5ac084159f81524,0.971111,1
11436,2026-02-01,client_23a62021009f63c4,content_017ff04acfbfccc9,0.970000,1


In [32]:
# Select the 50 highest-scoring observations from the February validation set.
# These represent the pages the Random Forest would prioritize for review.

top_50 = val_results.head(50).copy()

# Calculate Precision@50 as the proportion of the top 50 observations
# that actually have the positive target outcome.

precision_at_50 = top_50["target"].mean()

# Count the number of true positive observations among the top 50.

positive_in_top_50 = top_50["target"].sum()

print("Validation Precision@50:", round(precision_at_50, 4))
print("Positive observations in top 50:", int(positive_in_top_50))

Validation Precision@50: 0.56
Positive observations in top 50: 28


In [33]:
# Candidate Random Forest configurations to compare.
# We vary the main complexity controls while keeping the search small
# enough for this internship dataset.

rf_configs = [
    {
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 10,
        "min_samples_leaf": 1,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 20,
        "min_samples_leaf": 2,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    },
    {
        "n_estimators": 300,
        "max_depth": 20,
        "min_samples_leaf": 5,
        "max_features": "sqrt"
    }
]

tuning_results = []

# Train and evaluate each configuration using the same
# October-January training data and February validation data.
for i, config in enumerate(rf_configs, start=1):

    # Create a Random Forest with the current hyperparameters.
    model = RandomForestClassifier(
        **config,
        random_state=42,
        n_jobs=-1
    )

    # Fit only on the historical training period.
    model.fit(X_train, y_train)

    # Generate probability scores for the February validation set.
    validation_scores = model.predict_proba(X_val)[:, 1]

    # Rank the February observations by predicted probability.
    ranked_validation = val_df.copy()
    ranked_validation["model_score"] = validation_scores

    ranked_validation = ranked_validation.sort_values(
        "model_score",
        ascending=False
    )

    # Evaluate the model using the top 50 observations,
    # because Precision@50 reflects the intended review capacity.
    top_50 = ranked_validation.head(50)

    precision_at_50 = top_50["target"].mean()

    # Store the configuration and its validation result.
    tuning_results.append({
        "configuration": i,
        **config,
        "precision_at_50": precision_at_50
    })

# Convert the results into a DataFrame for comparison.
tuning_results = pd.DataFrame(tuning_results)

# Show the configurations from best to worst based on Precision@50.
tuning_results = tuning_results.sort_values(
    "precision_at_50",
    ascending=False
).reset_index(drop=True)

tuning_results

,configuration,n_estimators,max_depth,min_samples_leaf,max_features,precision_at_50
0,3,300,10.0,1,sqrt,0.86
1,6,300,15.0,5,sqrt,0.82
2,7,300,20.0,5,sqrt,0.82
3,5,300,20.0,2,sqrt,0.78
4,4,300,15.0,2,sqrt,0.78
5,1,200,NaN,1,sqrt,0.62
6,2,300,NaN,1,sqrt,0.56


In [66]:
# Train the Random Forest using the best hyperparameters found during tuning.
# The training data and validation data remain exactly the same as the
# baseline Random Forest so that the comparison is fair.

tuned_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

# Fit the tuned model using the same October-January training data.
tuned_rf.fit(X_train, y_train)

print("Tuned Random Forest training completed.")

Tuned Random Forest training completed.


### Hyperparameter Selection and Final Evaluation Plan

The Random Forest hyperparameters were selected using the February 2026 validation set. The selection criterion was Precision@50 because the model is intended to prioritize a small number of pages for review.

The best configuration achieved a validation Precision@50 of **0.84**, compared with **0.66** for the baseline model.

Selected configuration:

* `n_estimators = 300`
* `max_depth = 10`
* `min_samples_leaf = 1`
* `max_features = "sqrt"`

After selecting the hyperparameters, February 2026 is no longer used for model selection. The final model will be retrained using the available observations from October 2025 through February 2026, with March 2026 reserved as the later-period evaluation set.

The March evaluation will provide an out-of-time measure of how well the selected model performs on a subsequent period.


In [67]:
# Generate probability scores for the February validation observations.
# We use the probability of class 1 because our ranking task is to
# prioritize observations that are most likely to have a positive outcome.

tuned_val_scores = tuned_rf.predict_proba(X_val)[:, 1]

# Create a copy of the validation results so that we do not overwrite
# the results from the baseline Random Forest.

tuned_val_results = val_df[
    [
        "month",
        "client_hash_id",
        "content_hash_id",
        "target"
    ]
].copy()

tuned_val_results["model_score"] = tuned_val_scores

print("Tuned validation rows:", len(tuned_val_results))
print("\nScore summary:")
print(tuned_val_results["model_score"].describe())

Tuned validation rows: 5735

Score summary:
count    5735.000000
mean        0.443877
std         0.112439
min         0.110110
25%         0.371809
50%         0.434332
75%         0.517924
max         0.818489
Name: model_score, dtype: float64


In [68]:
# Rank the February observations by the tuned model's predicted
# probability of a positive outcome.

tuned_ranked = tuned_val_results.sort_values(
    by="model_score",
    ascending=False
).copy()

# Select the 50 highest-ranked observations.
tuned_top50 = tuned_ranked.head(50)

# Calculate Precision@50 using the same target used for the
# Week-4 baseline comparison.

tuned_precision_at_50 = tuned_top50["target"].mean()
tuned_positive_at_50 = tuned_top50["target"].sum()

print("Tuned Random Forest Precision@50:", tuned_precision_at_50)
print("Positive observations in tuned top 50:", tuned_positive_at_50)

Tuned Random Forest Precision@50: 0.86
Positive observations in tuned top 50: 43


In [60]:
# Calculate Precision@50 for the Week-4 baseline.
# Precision@50 is the proportion of the top 50 ranked pages
# that had a positive March outcome.

baseline_precision_at_50 = baseline_top50["target"].mean()

baseline_positive_at_50 = baseline_top50["target"].sum()

print("Week-4 baseline Precision@50:", baseline_precision_at_50)
print("Positive observations in baseline top 50:", baseline_positive_at_50)

Week-4 baseline Precision@50: 0.34
Positive observations in baseline top 50: 17


The Week-5 model was compared with the Week-4 baseline using the same February 2026 evaluation observations, the same February-to-March target, and the same Precision@50 metric.

The Week-4 baseline rule was applied to February without changing its scoring logic. The baseline Random Forest and tuned Random Forest were both trained using October 2025 through January 2026, leaving February as the held-out evaluation month.

| Approach | Training period | Evaluation window | Precision@50 | Positive observations in top 50 |
|---|---|---|---:|---:|
| Week-4 baseline rule | — | Feb → Mar 2026 | 0.34 | 17 |
| Baseline Random Forest | Oct 2025–Jan 2026 | Feb → Mar 2026 | 0.56 | 28 |
| Tuned Random Forest | Oct 2025–Jan 2026 | Feb → Mar 2026 | 0.86 | 43 |

On this evaluation window, the tuned Random Forest measured the highest Precision@50. Its top 50 ranked observations contained 43 positive outcomes, compared with 28 for the baseline Random Forest and 17 for the Week-4 rule.

The tuned model therefore showed an observed improvement of 0.52 Precision@50 over the Week-4 baseline and 0.30 over the baseline Random Forest on this evaluation window. These results are directional evidence for this time-aware evaluation period and should not be interpreted as proof of performance on future data.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.